# [2장 4강] 실습: 회귀 모델 비교와 하이퍼파라미터 튜닝

## 실습 목표

- 단일 Train/Test 분할과 K-Fold Cross Validation의 차이를 설명할 수 있다.
- 5-Fold Cross Validation으로 회귀 모델의 평균 성능과 편차를 확인할 수 있다.
- GridSearchCV로 Ridge와 Lasso의 `alpha`를 탐색할 수 있다.
- 최적 하이퍼파라미터와 교차검증 MSE·RMSE를 확인할 수 있다.
- MAE와 RMSE로 최종 모델의 Test 성능을 비교할 수 있다.

## 사용 데이터

- 파일: `insurance(2).csv`
- Feature: `age`, `sex`, `bmi`, `children`, `smoker`, `region`
- Label: `charges`

## 진행 방식

- 각 문제의 설명과 요구사항을 확인한 뒤 코드 셀을 작성합니다.
- Test Set은 최종 모델 평가에만 사용합니다.

## 실습 준비: 데이터 불러오기

Google Colab에서는 파일을 Google Drive의 `MyDrive`에 저장합니다. Jupyter Notebook에서는 노트북과 같은 폴더에 저장합니다.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path='/content/drive/MyDrive/insurance.csv'
except ModuleNotFoundError:
    file_path='insurance.csv'

insurance_df=pd.read_csv(file_path)
display(insurance_df.head())
print('데이터 크기:',insurance_df.shape)
insurance_df.info()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


데이터 크기: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


## 필수 1: 단일 분할과 5-Fold 교차검증 비교

### 문제 1-1: Ridge 모델의 교차검증 성능 확인하기

#### 문제 설명

한 번의 Train/Test 분할은 우연히 포함된 데이터에 따라 결과가 달라질 수 있습니다. Train Set 내부에서 Ridge 모델에 5-Fold Cross Validation을 적용하고 Fold별 RMSE와 평균·표준편차를 확인합니다.

#### 요구사항

1. 중복 행을 제거하고 X와 y를 분리합니다.
2. `random_state=42`로 Train Set 80%, Test Set 20%를 분리합니다.
3. 수치형 컬럼은 StandardScaler, 범주형 컬럼은 OneHotEncoder로 처리하는 ColumnTransformer를 만듭니다.
4. 전처리기와 `Ridge(alpha=1.0)`을 Pipeline으로 구성합니다.
5. `KFold(n_splits=5, shuffle=True, random_state=42)`를 만듭니다.
6. Train Set에서 `neg_mean_squared_error` 기준으로 교차검증합니다.
7. Fold별 RMSE, 평균 RMSE와 표준편차를 출력합니다.
8. 교차검증이 단일 분할보다 필요한 이유를 설명합니다.

#### 결과 해석 작성

> Fold별 RMSE가 서로 다른 이유는 무엇이며, 단일 분할 대신 교차검증 평균을 사용하면 어떤 장점이 있나요?

In [3]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
import numpy as np

# 1. 중복 행 제거
insurance_df = insurance_df.drop_duplicates()

# X와 y 분리
X = insurance_df.drop('charges', axis=1)
y = insurance_df['charges']

print('중복 제거 후 데이터 크기:', insurance_df.shape)


# 2. Train Set 80%, Test Set 20% 분리
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print('Train Set 크기:', X_train.shape)
print('Test Set 크기:', X_test.shape)


# 3. 수치형 / 범주형 컬럼 구분
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

print('수치형 컬럼:', list(numeric_cols))
print('범주형 컬럼:', list(categorical_cols))


# ColumnTransformer 생성
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)


# 4. 전처리기 + Ridge(alpha=1.0) Pipeline 구성
ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('ridge', Ridge(alpha=1.0))
    ]
)


# 5. KFold 5-Fold Cross Validation
kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# 6. Train Set에서 neg_mean_squared_error 기준 교차검증
cv_scores = cross_val_score(
    ridge_pipeline,
    X_train,
    y_train,
    cv=kf,
    scoring='neg_mean_squared_error'
)


# 7. MSE -> RMSE 변환
rmse_scores = np.sqrt(-cv_scores)


# Fold별 RMSE 출력
for i, rmse in enumerate(rmse_scores, start=1):
    print(f'Fold {i} RMSE: {rmse:.2f}')

# 평균 RMSE
mean_rmse = rmse_scores.mean()

# 표준편차
std_rmse = rmse_scores.std()

print(f'\n평균 RMSE: {mean_rmse:.2f}')
print(f'RMSE 표준편차: {std_rmse:.2f}')

중복 제거 후 데이터 크기: (1337, 7)
Train Set 크기: (1069, 6)
Test Set 크기: (268, 6)
수치형 컬럼: ['age', 'bmi', 'children']
범주형 컬럼: ['sex', 'smoker', 'region']
Fold 1 RMSE: 5847.00
Fold 2 RMSE: 6003.81
Fold 3 RMSE: 6136.42
Fold 4 RMSE: 6235.39
Fold 5 RMSE: 6395.62

평균 RMSE: 6123.65
RMSE 표준편차: 188.42


C:\Users\rkd76\AppData\Local\Temp\ipykernel_26444\3237186769.py:32: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=['object']).columns


## 필수 2: GridSearchCV로 Ridge와 Lasso 최적 alpha 탐색

### 문제 2-1: Ridge와 Lasso의 최적 alpha 찾기

#### 문제 설명

여러 `alpha` 후보를 5-Fold 교차검증으로 비교하여 평균 검증 MSE가 가장 낮은 Ridge와 Lasso 설정을 찾습니다.

#### 요구사항

1. Ridge와 Lasso Pipeline을 각각 만듭니다.
2. `alpha` 후보를 `[0.01, 0.1, 1, 10, 100]`으로 설정합니다.
3. Lasso에는 `max_iter=10000`을 적용합니다.
4. 같은 K-Fold와 `neg_mean_squared_error`를 사용하여 두 GridSearchCV를 학습합니다.
5. 각 모델의 최적 `alpha`를 출력합니다.
6. 음수 점수를 양수 MSE로 바꾸고 RMSE를 계산합니다.
7. Test Set을 사용하지 않고 alpha를 선택해야 하는 이유를 설명합니다.

#### 결과 해석 작성

> GridSearchCV에서 Test Set을 사용하지 않고 Train Set의 교차검증 결과로 alpha를 선택해야 하는 이유는 무엇인가요?

In [4]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge, Lasso
from sklearn.pipeline import Pipeline
import numpy as np


# 1. Ridge Pipeline
ridge_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('ridge', Ridge())
    ]
)


# Lasso Pipeline
lasso_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('lasso', Lasso(max_iter=10000))
    ]
)


# 2. alpha 후보 설정
alpha_values = [0.01, 0.1, 1, 10, 100]


# Ridge의 alpha 파라미터
ridge_param_grid = {
    'ridge__alpha': alpha_values
}


# Lasso의 alpha 파라미터
lasso_param_grid = {
    'lasso__alpha': alpha_values
}


# 4. Ridge GridSearchCV
ridge_grid = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=ridge_param_grid,
    cv=kf,
    scoring='neg_mean_squared_error'
)

ridge_grid.fit(X_train, y_train)


# Lasso GridSearchCV
lasso_grid = GridSearchCV(
    estimator=lasso_pipeline,
    param_grid=lasso_param_grid,
    cv=kf,
    scoring='neg_mean_squared_error'
)

lasso_grid.fit(X_train, y_train)


# 5. 최적 alpha 출력
print('Ridge 최적 alpha:', ridge_grid.best_params_['ridge__alpha'])
print('Lasso 최적 alpha:', lasso_grid.best_params_['lasso__alpha'])


# 6. 음수 MSE → 양수 MSE → RMSE 계산

# Ridge
ridge_best_mse = -ridge_grid.best_score_
ridge_best_rmse = np.sqrt(ridge_best_mse)

# Lasso
lasso_best_mse = -lasso_grid.best_score_
lasso_best_rmse = np.sqrt(lasso_best_mse)


print('\n[Ridge]')
print(f'최적 alpha: {ridge_grid.best_params_["ridge__alpha"]}')
print(f'평균 검증 MSE: {ridge_best_mse:.2f}')
print(f'평균 검증 RMSE: {ridge_best_rmse:.2f}')


print('\n[Lasso]')
print(f'최적 alpha: {lasso_grid.best_params_["lasso__alpha"]}')
print(f'평균 검증 MSE: {lasso_best_mse:.2f}')
print(f'평균 검증 RMSE: {lasso_best_rmse:.2f}')

Ridge 최적 alpha: 1
Lasso 최적 alpha: 100

[Ridge]
최적 alpha: 1
평균 검증 MSE: 37534565.79
평균 검증 RMSE: 6126.55

[Lasso]
최적 alpha: 100
평균 검증 MSE: 37501542.54
평균 검증 RMSE: 6123.85


## 필수 3: 기본 모델과 튜닝 모델의 최종 성능 비교

### 문제 3-1: Test Set에서 최종 모델 비교하기

#### 문제 설명

기본 Ridge와 GridSearchCV가 선택한 최적 Ridge·Lasso를 지금까지 사용하지 않은 Test Set에서 한 번 평가하여 실제 일반화 성능을 비교합니다.

#### 요구사항

1. 기본 `Ridge(alpha=1.0)` Pipeline을 전체 Train Set으로 학습합니다.
2. GridSearchCV의 `best_estimator_`에서 최적 Ridge와 Lasso를 가져옵니다.
3. 세 모델로 Test Set을 예측합니다.
4. MAE와 RMSE를 계산해 비교표를 만듭니다.
5. 가장 낮은 Test RMSE의 모델을 출력합니다.
6. 기본 Ridge 대비 튜닝 Ridge의 성능이 개선됐는지 설명합니다.
7. CV 성능이 가장 좋은 모델과 Test 성능이 가장 좋은 모델이 다를 수 있는 이유를 설명합니다.

#### 결과 해석 작성

> 튜닝 Ridge는 기본 Ridge보다 Test RMSE가 개선됐나요? 또한 CV RMSE가 가장 낮은 모델이 Test Set에서도 반드시 가장 좋아야 하나요?

## 심화 1: alpha별 교차검증 성능 변화 확인

### 문제 4-1: alpha별 RMSE 시각화하기

#### 문제 설명

GridSearchCV의 전체 결과를 표와 그래프로 정리하여 정규화 강도가 변할 때 평균 검증 RMSE가 어떻게 달라지는지 확인합니다.

#### 요구사항

1. Ridge와 Lasso의 `cv_results_`를 DataFrame으로 만듭니다.
2. 각 alpha의 음수 평균 검증 MSE를 RMSE로 변환합니다.
3. alpha별 RMSE 비교표를 출력합니다.
4. x축을 로그 스케일로 설정해 두 모델의 RMSE를 선 그래프로 비교합니다.
5. 후보 범위 안에서 최적 alpha의 RMSE가 가장 낮은지 확인합니다.
6. 정규화가 너무 강하면 성능이 나빠질 수 있는 이유를 설명합니다.

#### 결과 해석 작성

> alpha가 너무 커질 때 Ridge의 검증 RMSE가 크게 증가하는 이유는 무엇인가요?

## 실습 마무리

1. Test Set을 GridSearchCV에 사용하면 안 되는 이유는 무엇인가요?
2. Grid Search의 후보를 많이 지정할 때 발생하는 단점은 무엇인가요?
3. 최적 alpha가 후보 범위의 끝값이라면 무엇을 추가로 확인해야 하나요?